# Proxy Feature Audit

**Question:** if we delete the region column, can the other columns still reveal where an applicant lives?

**How we test it:** train a model to guess *Remote vs Centre* from the other columns, then measure how well it guesses.

**AUC in plain English:** 0.5 means random guessing (no regional signal). 1.0 means it tells Remote from Centre perfectly (full leak).

**Steps**
1. Each column alone
2. Drop one column at a time
3. Grid search over every combination of columns
4. The three groups: hidden proxies, largest safe sets, minimal leaking sets

This is an association check, not proof that a column causes unfair decisions.

## 0. Setup

Loads the 10,000 historical applications and builds the target (1 = Remote, 0 = Centre).
The region column itself is **not** a feature. Same Random Forest, same 70/30 split for every test.

Two settings to change if needed:
- `TOP_N`: how many rows to show in each list.
- `RECOMPUTE_GRID`: set to `False` to reload the saved grid instead of retraining 127 models.

In [2]:
from itertools import combinations
from pathlib import Path

import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

TOP_N = 5               # rows shown per list
RECOMPUTE_GRID = True   # False = reload proxy_grid.csv instead of retraining 127 models

pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", 250)

# Find the repo root by walking up until the data file is found
raw_relative_path = Path("equialgo-participants/data/donnees_demandes.csv")
repository_root = next(
    (p for p in (Path.cwd(), *Path.cwd().parents) if (p / raw_relative_path).is_file()),
    None,
)
if repository_root is None:
    raise FileNotFoundError(f"Could not locate {raw_relative_path} from {Path.cwd()}")

OUT_DIR = repository_root / "Analysis" / "proxy_analysis" / "proxy_outputs"
OUT_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(repository_root / raw_relative_path)

# Target: 1 = Remote region, 0 = Centre
REMOTE = {"Bas-Saint-Laurent", "Cote-Nord", "Gaspesie-Iles-de-la-Madeleine"}
y = df["region_administrative"].isin(REMOTE).astype(int)

# Features: everything except the region itself, the ID and the committee decision
num = [
    "cote_r_equivalent",
    "revenu_familial_estime",
    "heures_travail_semaine",
    "distance_domicile_campus_km",
    "premiere_generation_universitaire",
]
cat = ["programme_etudes", "code_postal_3"]
all_cols = num + cat

X_tr, X_te, y_tr, y_te = train_test_split(
    df[all_cols], y, test_size=0.3, random_state=42, stratify=y
)


def auc_with(cols):
    """Train on the selected columns and return the test AUC for predicting Remote."""
    cats = [c for c in cols if c in cat]
    nums = [c for c in cols if c in num]
    prep = ColumnTransformer(
        [("cat", OneHotEncoder(handle_unknown="ignore"), cats)] if cats else [],
        remainder="passthrough",
    )
    model = Pipeline([
        ("prep", prep),
        ("rf", RandomForestClassifier(
            n_estimators=300, min_samples_leaf=20, random_state=42, n_jobs=-1
        )),
    ])
    selected = cats + nums
    model.fit(X_tr[selected], y_tr)
    return roc_auc_score(y_te, model.predict_proba(X_te[selected])[:, 1])


def strength(auc):
    """Plain-English label for an AUC."""
    if auc >= 0.95:
        return "near-perfect proxy"
    if auc >= 0.75:
        return "strong proxy"
    if auc >= 0.60:
        return "moderate proxy"
    return "weak / not a proxy"

## 1. Each column alone

Train the model on **one column only** and see how well it guesses the region.
A high AUC means that column gives away the region by itself.

In [3]:
single = pd.DataFrame(
    [{"column": c, "auc": auc_with([c])} for c in all_cols]
).sort_values("auc", ascending=False).reset_index(drop=True)
single["strength"] = single["auc"].apply(strength)

print(single.round(4).to_string(index=False))
single.round(4).to_csv(OUT_DIR / "proxy_single_column.csv", index=False)

                           column    auc           strength
                    code_postal_3 1.0000 near-perfect proxy
      distance_domicile_campus_km 0.9979 near-perfect proxy
           heures_travail_semaine 0.8015       strong proxy
           revenu_familial_estime 0.6688     moderate proxy
premiere_generation_universitaire 0.5831 weak / not a proxy
                 programme_etudes 0.5567 weak / not a proxy
                cote_r_equivalent 0.5139 weak / not a proxy


## 2. Drop one column at a time

Train on **all columns**, then remove one column and see how much the AUC falls.
A big drop means the model needed that column. A tiny drop means the other columns cover for it,
which does **not** mean the column is harmless.

In [4]:
full_auc = auc_with(all_cols)
rows = [{"column": "ALL columns", "auc": full_auc, "auc_drop": 0.0}]

for c in all_cols:
    a = auc_with([x for x in all_cols if x != c])
    rows.append({"column": f"without {c}", "auc": a, "auc_drop": full_auc - a})

drop_one = pd.DataFrame(rows)
print(drop_one.round(4).to_string(index=False))
drop_one.round(4).to_csv(OUT_DIR / "proxy_drop_one.csv", index=False)

                                   column    auc  auc_drop
                              ALL columns 0.9998    0.0000
                without cote_r_equivalent 0.9999   -0.0000
           without revenu_familial_estime 0.9999   -0.0000
           without heures_travail_semaine 0.9998    0.0000
      without distance_domicile_campus_km 1.0000   -0.0002
without premiere_generation_universitaire 0.9998    0.0000
                 without programme_etudes 0.9999   -0.0000
                    without code_postal_3 0.9983    0.0015


### 2b. Drop several columns together

Remove the strongest carriers step by step to see how much signal is left.

In [5]:
group_drops = [
    ["code_postal_3", "distance_domicile_campus_km"],
    ["code_postal_3", "distance_domicile_campus_km", "revenu_familial_estime"],
    ["code_postal_3", "distance_domicile_campus_km", "revenu_familial_estime",
     "heures_travail_semaine"],
]

rows = []
for drop in group_drops:
    keep = [c for c in all_cols if c not in drop]
    rows.append({"removed": ", ".join(drop), "n_removed": len(drop), "auc_left": auc_with(keep)})

drop_groups = pd.DataFrame(rows)
print(drop_groups.round(4).to_string(index=False))
drop_groups.round(4).to_csv(OUT_DIR / "proxy_drop_groups.csv", index=False)

                                                                                   removed  n_removed  auc_left
                                                code_postal_3, distance_domicile_campus_km          2    0.8414
                        code_postal_3, distance_domicile_campus_km, revenu_familial_estime          3    0.8106
code_postal_3, distance_domicile_campus_km, revenu_familial_estime, heures_travail_semaine          4    0.5967


## 3. Grid search over every combination

With 7 columns there are 2^7 - 1 = **127** non-empty combinations. We train one model per combination
and record its AUC. This shows which sets of columns leak the region and which stay close to random.

It takes a few minutes. Results are saved to `proxy_grid.csv`.

In [6]:
grid_path = OUT_DIR / "proxy_grid.csv"

if RECOMPUTE_GRID or not grid_path.is_file():
    total = 2 ** len(all_cols) - 1
    grid_rows = []
    for size in range(1, len(all_cols) + 1):
        for subset in combinations(all_cols, size):
            grid_rows.append({
                "columns": ", ".join(subset),
                "n": size,
                "auc": auc_with(list(subset)),
            })
            if len(grid_rows) % 10 == 0 or len(grid_rows) == total:
                print(f"{len(grid_rows)}/{total} done")
    grid = pd.DataFrame(grid_rows).sort_values("auc", ascending=False).reset_index(drop=True)
    grid.round(4).to_csv(grid_path, index=False)
else:
    grid = pd.read_csv(grid_path)

print(f"\n{len(grid)} combinations. Saved to {grid_path}")

10/127 done
20/127 done
30/127 done
40/127 done
50/127 done
60/127 done
70/127 done
80/127 done
90/127 done
100/127 done
110/127 done
120/127 done
127/127 done

127 combinations. Saved to /Users/nawarturk/Desktop/1_Projects/polyml_hack/ivado/Analysis/proxy_analysis/proxy_outputs/proxy_grid.csv


## 4. The three groups

Top few of each list (set `TOP_N` in the setup cell). Postal code and distance leak the region on their own,
so any set that contains one of them scores about 1.0 and says nothing new. The lists below remove that noise:

1. **Hidden proxies:** the strongest leaks among sets that do **not** use postal code or distance.
   This shows how much region signal is left in the other columns.
2. **Largest safe sets:** the biggest sets of columns that still stay close to random (AUC below 0.6).
   These are candidate model inputs.
3. **Minimal leaking sets:** sets with AUC above 0.95 where no smaller part already leaks.
   This shows the true minimum culprits.

In [7]:
grid = grid.copy()
grid["auc"] = grid["auc"].round(4)
col_sets = grid["columns"].str.split(", ").apply(frozenset)
OBVIOUS = {"code_postal_3", "distance_domicile_campus_km"}

# 1. Hidden proxies: strongest leaks without postal code or distance
no_obvious = grid[col_sets.apply(lambda s: not (s & OBVIOUS))]
hidden_proxies = no_obvious.sort_values(["auc", "n"], ascending=[False, True]).head(TOP_N)

# 2. Largest safe sets: most columns while AUC stays below 0.6
safe_sets = (
    grid[grid["auc"] < 0.6]
    .sort_values(["n", "auc"], ascending=[False, True])
    .head(TOP_N)
)

# 3. Minimal leaking sets: AUC > 0.95 and no smaller subset already leaks
leaking = grid[grid["auc"] > 0.95]
leaking_sets = list(col_sets[leaking.index])
is_minimal = [not any(other < s for other in leaking_sets) for s in leaking_sets]
minimal_leaking = (
    leaking[is_minimal]
    .sort_values(["n", "auc"], ascending=[True, False])
    .head(TOP_N)
)

for title, table, filename in [
    ("HIDDEN PROXIES (strongest leaks without postal code or distance)",
     hidden_proxies, "proxy_hidden_proxies.csv"),
    ("LARGEST SAFE SETS (most columns, AUC < 0.6)", safe_sets, "proxy_safe_sets.csv"),
    ("MINIMAL LEAKING SETS (AUC > 0.95, no smaller subset leaks)",
     minimal_leaking, "proxy_minimal_leaking.csv"),
]:
    print(f"\n{title}")
    print(table.round(4).to_string(index=False))
    table.round(4).to_csv(OUT_DIR / filename, index=False)

print(f"\nBest AUC without postal code or distance: {no_obvious['auc'].max():.4f}")
print(f"All tables saved in {OUT_DIR}")


HIDDEN PROXIES (strongest leaks without postal code or distance)
                                                                                                               columns  n    auc
cote_r_equivalent, revenu_familial_estime, heures_travail_semaine, premiere_generation_universitaire, programme_etudes  5 0.8414
                   revenu_familial_estime, heures_travail_semaine, premiere_generation_universitaire, programme_etudes  4 0.8388
                  cote_r_equivalent, revenu_familial_estime, heures_travail_semaine, premiere_generation_universitaire  4 0.8370
                                   cote_r_equivalent, revenu_familial_estime, heures_travail_semaine, programme_etudes  4 0.8335
                                                      revenu_familial_estime, heures_travail_semaine, programme_etudes  3 0.8319

LARGEST SAFE SETS (most columns, AUC < 0.6)
                                                               columns  n    auc
cote_r_equivalent, premiere_genera